# Two-parameter persistence with pyDowker and multipers

This notebook demonstrates the bifiltration workflow: build a bifiltered Dowker complex
and compute a module approximation of its two-parameter persistent homology using
[multipers](https://github.com/DavidLapous/multipers).

Install: `pip install pyDowker[multipers] scikit-learn`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyDowker import DowkerComplex
from sklearn.metrics import pairwise_distances
import multipers as mp

## Two small point clouds

We take two sets of four points in the plane, $X$ (data) and $Y$ (witnesses),
and form the relation matrix $\Lambda_{ij} = d(x_i, y_j)$.

In [ ]:
X = np.array([(0, 0), (4, 0), (0, 6), (4, 3)])
Y = np.array([(1, 3), (3, 3), (3, 1), (1, 1)])

f, ax = plt.subplots()
ax.scatter(X[:, 0], X[:, 1], label='$X$ (data)')
ax.scatter(Y[:, 0], Y[:, 1], label='$Y$ (witnesses)')
ax.legend()
plt.show()

## Bifiltration and module approximation

The bifiltered Dowker complex $\mathcal{D}(X, Y, R_{\bullet})$ is parameterised by

- **Axis 1** ($r$): the sublevel threshold — a simplex $\sigma$ appears once all
  its data points have a witness within distance $r$.
- **Axis 2** ($-m$): the (negated) witness count — $\sigma$ persists as long as at
  least $m$ common witnesses exist.

`create_multipers_bifiltration` returns a `multipers.SimplexTreeMulti` (k-critical,
2 parameters) ready for module approximation.

In [ ]:
Dow_XY = DowkerComplex(pairwise_distances(X, Y))
st = Dow_XY.create_multipers_bifiltration(max_dimension=2, m_max=4)

mma = mp.module_approximation(st)

# Axis ranges: distance ∈ [0, 8], witness-count ∈ [-4, 0]
mma.plot(degree=0, box=[[0, -4], [8, 0]])
plt.title('Degree-0 module approximation')
plt.show()

## Correctness check: Python vs C++ implementation

Both `create_simplex_tree` (Python) and `create_simplex_tree_cpp` (C++) should give
identical filtration values for the sublevel filtration.  We verify this on a random
80×10 matrix.

In [ ]:
rng = np.random.default_rng(0)
rel = rng.random((80, 10))

dc = DowkerComplex(rel)
st_py  = dc.create_simplex_tree(filtration='Sublevel', max_dimension=2, m=1)
st_cpp = dc.create_simplex_tree_cpp(filtration='Sublevel', max_dimension=2, m=1)

mismatches = [
    (s, f, st_cpp.filtration(s))
    for s, f in st_py.get_filtration()
    if f != st_cpp.filtration(s)
]
print('Mismatches:', mismatches if mismatches else 'none — implementations agree ✓')

## Bifiltration on a worked example

Using the 5×4 relation from the paper (Figure 4), we build the full bifiltration
and compute the module approximation of its degree-1 persistent homology.

In [ ]:
relation = np.array(
    [[1, 2, 3, 4],
     [2, 3, 4, 5],
     [2, 0, 2, 6],
     [4, 7, 1, 1],
     [3, 1, 4, 1]], dtype=np.float64
)

dc2 = DowkerComplex(relation)
stmp = dc2.create_multipers_bifiltration(max_dimension=3, m_max=4)

mma2 = mp.module_approximation(stmp)
mma2.plot(degree=1, box=[[0, -4], [8, 0]])
plt.title('Degree-1 module approximation')
plt.show()